# Full experiments

Run this notebook beside the solver files and `data/`.
Ablation uses gamma = 0.1 and omega = 0.9.


In [ ]:
from __future__ import annotations

import contextlib
import copy
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import inspect
import io
import json
import math
from pathlib import Path
import platform
import random
import shutil
import sys
import time
import uuid
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

# Reload modules to avoid using an earlier kernel import.
importlib.invalidate_caches()
import frvcp_dwc as charging_module

importlib.reload(charging_module)
import tabu_search_impl as search_module

importlib.reload(search_module)
from frvcpy import core, translator

DwcFrvcpSolver = search_module.DwcFrvcpSolver
RouteEvaluationCache = search_module.RouteEvaluationCache
TabuSearch = search_module.TabuSearch
TabuSearchResult = search_module.TabuSearchResult
build_initial_solution = search_module.build_initial_solution
compute_nearest_neighbors = search_module.compute_nearest_neighbors
multi_start_tabu_search = search_module.multi_start_tabu_search

# -------------------------
# General configuration
# -------------------------
DATA_DIR = Path("data")
OUTPUT_PARENT = Path("runs")
RESUME_OUTPUT_DIR: Optional[str] = (
    None  # Set to a matching runs/run_... directory to resume.
)

INSTANCE_LIMIT: Optional[int] = None  # None selects all 60 benchmarks.
INSTANCE_FILTER: Optional[Set[str]] = None
INSTANCE_SIZES = {10, 20, 40}

COVERAGE_LEVELS = [0.10, 0.20, 0.30]
BASE_OMEGA = 0.9
OMEGA_LEVELS = [0.3, 0.5, 0.7, 0.9]
BASE_SOLVER_SEED = 20240528
MULTISEED_PLACEMENT_SEEDS = [20240528 + i for i in range(5)]
SENSITIVITY_PLACEMENT_SEED = 20240528
ABLATION_PLACEMENT_SEED = 20240528
ABLATION_CASES = [(0.10, 0.9)]

RUN_NO_WIRELESS_BASELINE = True
RUN_MULTI_SEED = True
RUN_SENSITIVITY = True
RUN_ABLATION = True

USE_MULTI_START = True
NUM_RUNS = 5
MAX_ITERATIONS = 100
MAX_MOVES_PER_ITERATION = 1200
NEIGHBOR_LIST_SIZE = 20
BASE_TABU_TENURE = 5
SHAKE_TENURE = 20
SHAKE_STEPS = 1
MAX_SHAKE_TRIALS = 10
OPT_ITER = 0

DEPOT_CHARGING_ALLOWED = True
INITIAL_SOC_FRACTION = 1.0
MULTI_INSERT_ALLOWED = True
DEPOT_ID = 0
QUIET_TRANSLATOR_LOGS = True
QUIET_SOLVER_LOGS = True
SAVE_SEARCH_TRACES = True
FAIL_FAST = True  # Checkpoint each completed case; stop on the first error.

SEARCH_VERSION = "multistart-interroute-reset-shaking-20261026"
ENERGY_MODEL_VERSION = "linear-dwc-v2"
EVALUATOR_VERSION = "dwc-bounds-v1"
EXPECTED_SEARCH_SHA256 = (
    "503c695edc152f017aeef1b18f4334339c9113fcfd27b64a175110fe2b4e4054"
)


def source_sha256(path):
    # Normalize Windows and Unix line endings before hashing source.
    return hashlib.sha256(Path(path).read_text(encoding="utf-8").encode()).hexdigest()


SEARCH_SHA256 = source_sha256(search_module.__file__)
CHARGING_SHA256 = source_sha256(charging_module.__file__)
if SEARCH_SHA256 != EXPECTED_SEARCH_SHA256:
    raise RuntimeError(
        "Search source differs from the published campaign. "
        "After intentional algorithm changes, update the expected hash and run a new campaign."
    )
required_options = {"shake_steps", "max_shake_trials", "shake_inter_route_only"}
if not required_options.issubset(inspect.signature(TabuSearch.__init__).parameters):
    raise RuntimeError("The imported search module lacks the required shaking options.")
print("Search module:", Path(search_module.__file__).resolve())
print("Charging module:", Path(charging_module.__file__).resolve())
print(
    f"Budget: {NUM_RUNS if USE_MULTI_START else 1} starts x {MAX_ITERATIONS} TS iterations; "
    f"at most {MAX_MOVES_PER_ITERATION} sampled candidates/iteration, plus shaking."
)


In [ ]:
def _extract_n_from_name(name: str) -> Optional[int]:
    import re

    # Parse benchmark names like tc0c10s2cf1.xml -> n_customers = 10.
    m = re.search(r"tc\d+c(\d+)", name)
    if m:
        return int(m.group(1))

    # Fallback for alternate naming conventions.
    m = re.search(r"c(\d+)s", name)
    return int(m.group(1)) if m else None


def collect_instances(
    data_dir: Path,
    limit: Optional[int] = None,
    include_names: Optional[Set[str]] = None,
    allowed_sizes: Optional[Set[int]] = None,
) -> List[Path]:
    paths = list(data_dir.glob("*.xml"))
    if include_names:
        paths = [p for p in paths if p.name in include_names]
    if allowed_sizes:
        paths = [p for p in paths if _extract_n_from_name(p.name) in allowed_sizes]

    # Force stable order by size first: c10 -> c20 -> c40, then by filename.
    paths = sorted(
        paths, key=lambda p: ((_extract_n_from_name(p.name) or 10**9), p.name)
    )

    if limit is not None:
        paths = paths[:limit]
    if not paths:
        raise RuntimeError("No instance found with current filters")
    return paths


def paper_tabu_params(
    n_customers: int, rng_seed: int = BASE_SOLVER_SEED
) -> Dict[str, Any]:
    # shake_iter/phi are retained by the API; tenure adaptation is disabled.
    return dict(
        base_tabu_tenure=BASE_TABU_TENURE,
        max_iter=MAX_ITERATIONS,
        shake_tenure=SHAKE_TENURE,
        shake_iter=20,
        opt_iter=OPT_ITER,
        phi=0.9,
        rng_seed=int(rng_seed),
        use_candidate_lists=True,
        use_extended_operators=True,
        shake_steps=SHAKE_STEPS,
        max_shake_trials=MAX_SHAKE_TRIALS,
        shake_inter_route_only=True,
        candidate_list_size=None,
    )


def translate_instance(xml_path: Path) -> Dict[str, Any]:
    if QUIET_TRANSLATOR_LOGS:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(
            io.StringIO()
        ):
            return translator.translate(
                xml_path.as_posix(), depot_charging=DEPOT_CHARGING_ALLOWED
            )
    return translator.translate(
        xml_path.as_posix(), depot_charging=DEPOT_CHARGING_ALLOWED
    )


def extract_customers(instance_dict: Dict[str, Any]) -> List[int]:
    inst = core.FrvcpInstance(copy.deepcopy(instance_dict))
    return [
        node.node_id
        for node in inst.nodes_g
        if node.node_id != DEPOT_ID and node.type == core.NodeType.CUSTOMER
    ]


def _instance_selection_seed(instance_name: str, base_seed: int) -> int:
    digest = hashlib.sha256(f"{instance_name}|{base_seed}".encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % (2**32)


def _customer_candidate_undirected_pairs(
    instance_dict: Dict[str, Any],
) -> List[Tuple[int, int]]:
    customer_ids = sorted(extract_customers(instance_dict))
    return [
        (customer_ids[a], customer_ids[b])
        for a in range(len(customer_ids))
        for b in range(a + 1, len(customer_ids))
    ]


def build_incremental_arc_plan(
    instance_dict: Dict[str, Any],
    instance_name: str,
    coverage_levels: Sequence[float],
    base_seed: int,
) -> Dict[str, Any]:
    undirected_pairs = _customer_candidate_undirected_pairs(instance_dict)
    if not undirected_pairs:
        return {
            "candidate_pairs": [],
            "ordered_pairs": [],
            "coverages": list(coverage_levels),
            "selections": [[] for _ in coverage_levels],
        }

    rng = random.Random(_instance_selection_seed(instance_name, base_seed))
    ordered_pairs = undirected_pairs[:]
    rng.shuffle(ordered_pairs)

    selections: List[List[Tuple[int, int]]] = []
    total_pairs = len(ordered_pairs)
    for cov in coverage_levels:
        if cov <= 0.0:
            selections.append([])
            continue
        pair_count = max(1, int(total_pairs * cov))
        pair_count = min(pair_count, total_pairs)
        selected_pairs = ordered_pairs[:pair_count]
        directed: List[Tuple[int, int]] = []
        for i, j in selected_pairs:
            directed.append((i, j))
            directed.append((j, i))
        selections.append(directed)

    return {
        "candidate_pairs": undirected_pairs,
        "ordered_pairs": ordered_pairs,
        "coverages": list(coverage_levels),
        "selections": selections,
    }


def solve_instance(
    instance_dict: Dict[str, Any],
    solver_kwargs: Dict[str, Any],
) -> TabuSearchResult:
    frvcp_instance = core.FrvcpInstance(copy.deepcopy(instance_dict))
    customers = [
        node.node_id
        for node in frvcp_instance.nodes_g
        if node.node_id != DEPOT_ID and node.type == core.NodeType.CUSTOMER
    ]
    if not customers:
        raise RuntimeError("No customer in translated instance")

    effective_kwargs = dict(solver_kwargs)
    # Policy: no candidate filtering for c10/c20
    if len(customers) <= 20:
        effective_kwargs["use_candidate_lists"] = False

    if not np.isfinite(INITIAL_SOC_FRACTION) or not 0 <= INITIAL_SOC_FRACTION <= 1:
        raise ValueError("INITIAL_SOC_FRACTION must lie in [0, 1]")
    initial_soc = frvcp_instance.max_q * INITIAL_SOC_FRACTION
    cache = RouteEvaluationCache(
        instance_dict,
        frvcp_instance,
        initial_soc,
        allow_multi_insert=MULTI_INSERT_ALLOWED,
    )

    use_candidate_lists = bool(effective_kwargs.get("use_candidate_lists", False))
    neighbor_map = (
        compute_nearest_neighbors(frvcp_instance, customers, NEIGHBOR_LIST_SIZE)
        if use_candidate_lists
        else {}
    )

    if USE_MULTI_START:
        return multi_start_tabu_search(
            customers,
            cache,
            required_customers=set(customers),
            num_runs=NUM_RUNS,
            neighbor_map=neighbor_map,
            max_moves_per_iter=MAX_MOVES_PER_ITERATION,
            **effective_kwargs,
        )

    rng = random.Random(effective_kwargs["rng_seed"])
    with cache.evaluation_phase("construction"):
        initial_routes = build_initial_solution(
            customers, cache, rng, randomized=True, rcl_size=5
        )
    search = TabuSearch(
        cache=cache,
        all_customers=set(customers),
        neighbor_map=neighbor_map,
        max_moves_per_iter=MAX_MOVES_PER_ITERATION,
        **effective_kwargs,
    )
    return search.run(initial_routes)


def apply_wireless_overlay(base_instance, selected_arcs, omega):
    """Battery-side traction-energy recovery; no energy floor or energy gain."""
    omega = float(omega)
    if not np.isfinite(omega) or not 0 <= omega <= 1:
        raise ValueError("omega must be finite and lie in [0, 1]")
    instance = copy.deepcopy(base_instance)
    energy = np.asarray(instance["energy_matrix"], dtype=float).copy()
    if energy.ndim != 2 or energy.shape[0] != energy.shape[1]:
        raise ValueError("Energy matrix must be square")
    if not np.isfinite(energy).all() or (energy < 0).any():
        raise ValueError("Base energy matrix must be finite and nonnegative")
    customers = set(extract_customers(base_instance))
    arcs = sorted({(int(i), int(j)) for i, j in selected_arcs})
    reduction = 0.0
    for i, j in arcs:
        if i == j or i not in customers or j not in customers:
            raise ValueError("Only arcs between distinct customers can be equipped")
        before = float(energy[i, j])
        energy[i, j] = (1.0 - omega) * before
        reduction += before - float(energy[i, j])
    instance["energy_matrix"] = energy.tolist()
    return instance, dict(
        selected_arcs=len(arcs), applied_arcs=len(arcs), total_reduction=reduction
    )


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (set, tuple)):
        return sorted(value) if isinstance(value, set) else list(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")


def write_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            value, ensure_ascii=False, indent=2, default=json_default, allow_nan=False
        ),
        encoding="utf-8",
    )
    temporary.replace(path)


def write_csv_atomic(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def hash_object(value):
    encoded = json.dumps(value, sort_keys=True, default=json_default, allow_nan=False)
    return hashlib.sha256(encoded.encode()).hexdigest()


@contextlib.contextmanager
def capture_start_results():
    """Record all starts without altering RNG, route evaluation, or selection.

    The notebook runs cases sequentially. Restore the method even on failure.
    Copy stats before multi_start_tabu_search aggregates the winning result.
    """
    original_run = TabuSearch.run
    original_shake = TabuSearch._shake
    records = []
    shake_events = {}

    def recorded_shake(search, routes):
        before = copy.deepcopy(routes)
        outcome = original_shake(search, routes)
        after, cost, details = outcome
        shake_events.setdefault(id(search), []).append(
            dict(
                iteration=len(search.history),
                before_routes=before,
                after_routes=copy.deepcopy(after),
                shaken_cost=float(cost) if math.isfinite(cost) else None,
                feasible=details is not None and math.isfinite(cost),
                changed=(
                    search_module.canonical_solution_signature(before)
                    != search_module.canonical_solution_signature(after)
                ),
            )
        )
        return outcome

    def recorded_run(search, initial_routes):
        shake_events[id(search)] = []
        result = original_run(search, initial_routes)
        records.append(
            dict(
                start_index=len(records),
                best_cost=float(result.best_cost),
                vehicles=len(result.routes),
                routes=copy.deepcopy(result.routes),
                stats=copy.deepcopy(result.stats),
                history=copy.deepcopy(result.history),
                shake_events=shake_events.pop(id(search)),
            )
        )
        return result

    TabuSearch.run = recorded_run
    TabuSearch._shake = recorded_shake
    try:
        yield records
    finally:
        TabuSearch.run = original_run
        TabuSearch._shake = original_shake


def effective_solver_kwargs(n_customers, solver_seed, overrides=None):
    kwargs = paper_tabu_params(n_customers, solver_seed)
    kwargs.update(overrides or {})
    if n_customers <= 20:
        kwargs["use_candidate_lists"] = False
    return kwargs


def protocol_settings(n_customers, solver_seed, overrides=None):
    return dict(
        solver_kwargs=effective_solver_kwargs(n_customers, solver_seed, overrides),
        use_multi_start=USE_MULTI_START,
        num_runs=NUM_RUNS if USE_MULTI_START else 1,
        neighbor_list_size=NEIGHBOR_LIST_SIZE,
        max_moves_per_iteration=MAX_MOVES_PER_ITERATION,
        initial_soc_fraction=INITIAL_SOC_FRACTION,
        multi_insert_allowed=MULTI_INSERT_ALLOWED,
        depot_charging_allowed=DEPOT_CHARGING_ALLOWED,
        search_version=SEARCH_VERSION,
        evaluator_version=EVALUATOR_VERSION,
        energy_model_version=ENERGY_MODEL_VERSION,
        search_sha256=SEARCH_SHA256,
        charging_sha256=CHARGING_SHA256,
    )


def solve_overlay_case(case):
    path = Path(case["instance_path"])
    base = translate_instance(path)
    customers = extract_customers(base)
    n = len(customers)
    named_size = _extract_n_from_name(path.name)
    if named_size is not None and named_size != n:
        raise ValueError(
            f"{path.name}: name says {named_size} customers; translation gives {n}"
        )
    plan = build_incremental_arc_plan(
        base, path.name, [case["coverage"]], case["placement_seed"]
    )
    arcs = plan["selections"][0]
    instance, overlay = apply_wireless_overlay(base, arcs, case["omega"])
    kwargs = effective_solver_kwargs(n, case["solver_seed"], case.get("overrides"))
    settings = protocol_settings(n, case["solver_seed"], case.get("overrides"))

    started = time.perf_counter()
    with capture_start_results() as starts:
        if QUIET_SOLVER_LOGS:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(
                io.StringIO()
            ):
                result = solve_instance(instance, kwargs)
        else:
            result = solve_instance(instance, kwargs)
    elapsed = time.perf_counter() - started
    if not np.isfinite(result.best_cost):
        raise RuntimeError("Solver returned a non-finite best cost")
    expected_starts = NUM_RUNS if USE_MULTI_START else 1
    if len(starts) != expected_starts:
        raise RuntimeError(
            f"Expected {expected_starts} start results, received {len(starts)}"
        )
    if sorted(c for route in result.routes for c in route) != sorted(customers):
        raise RuntimeError("Returned solution has missing or duplicate customers")
    if abs(min(s["best_cost"] for s in starts) - result.best_cost) > 1e-6:
        raise RuntimeError(
            "Returned best cost does not match the best of recorded starts"
        )

    stats = result.stats
    seeds = stats.get("start_seeds", [int(case["solver_seed"])])
    if len(seeds) != len(starts):
        raise RuntimeError("Start seeds and recorded starts have different lengths")
    for record, seed in zip(starts, seeds):
        record["start_seed"] = int(seed)

    row = dict(
        campaign_id=CAMPAIGN_ID,
        execution_id=case["case_id"],
        case_id=case["case_id"],
        experiment=case["experiment"],
        ablation=case["ablation"],
        protocol_id=hash_object(settings),
        protocol_json=json.dumps(settings, sort_keys=True),
        search_version=SEARCH_VERSION,
        energy_model_version=ENERGY_MODEL_VERSION,
        evaluator_version=EVALUATOR_VERSION,
        search_sha256=SEARCH_SHA256,
        charging_sha256=CHARGING_SHA256,
        instance=path.name,
        n_customers=n,
        input_sha256=INPUT_SHA256[path.name],
        coverage=case["coverage"],
        coverage_pct=f"{case['coverage']*100:g}%",
        omega=case["omega"],
        placement_seed=case["placement_seed"],
        solver_seed=case["solver_seed"],
        use_multi_start=USE_MULTI_START,
        num_runs=expected_starts,
        max_moves_per_iteration=MAX_MOVES_PER_ITERATION,
        neighbor_list_size=NEIGHBOR_LIST_SIZE,
        initial_soc_fraction=INITIAL_SOC_FRACTION,
        multi_insert_allowed=MULTI_INSERT_ALLOWED,
        best_cost=float(result.best_cost),
        vehicles=len(result.routes),
        mean_start_cost=float(np.mean([s["best_cost"] for s in starts])),
        std_start_cost=(
            float(np.std([s["best_cost"] for s in starts], ddof=1))
            if len(starts) > 1
            else None
        ),
        iterations_done=stats.get("iterations_total", 0),
        best_start_iterations=stats.get("best_start_iterations", 0),
        starts_completed=stats.get("starts_completed", 1),
        start_seeds_json=json.dumps(seeds),
        best_start_seed=stats.get("best_start_seed", seeds[0]),
        routes_json=json.dumps(result.routes),
        charging_plans_json=json.dumps(
            [detail.plan for detail in result.details], default=json_default
        ),
        start_results_json=json.dumps(
            [
                {
                    k: v
                    for k, v in s.items()
                    if k not in {"history", "routes", "shake_events"}
                }
                for s in starts
            ],
            default=json_default,
        ),
        solver_time_sec=elapsed,
        candidate_pairs=len(plan["candidate_pairs"]),
        selected_arcs=overlay["selected_arcs"],
        applied_arcs=overlay["applied_arcs"],
        total_reduction=overlay["total_reduction"],
        error=None,
    )
    for name in [
        "base_tabu_tenure",
        "max_iter",
        "shake_tenure",
        "shake_iter",
        "shake_steps",
        "max_shake_trials",
        "opt_iter",
        "phi",
        "use_candidate_lists",
        "use_extended_operators",
        "shake_inter_route_only",
        "candidate_list_size",
    ]:
        row[name] = kwargs[name]
    row["use_shaking"] = kwargs["shake_tenure"] <= kwargs["max_iter"]
    row["shaking_disabled_by_threshold"] = kwargs["shake_tenure"] > kwargs["max_iter"]
    # Built-in counters use the same definitions in every experimental variant.
    for key, value in stats.items():
        if key in search_module.SEARCH_COUNTER_KEYS or key.startswith("frvcp_"):
            row[key] = json_default(value) if isinstance(value, np.generic) else value
    calls = row.get("frvcp_calls", 0)
    charge_time = row.get("frvcp_total_time_sec", 0.0)
    row["frvcp_avg_time_ms"] = 1000.0 * charge_time / calls if calls else None
    row["frvcp_search_calls_per_iteration"] = (
        row.get("frvcp_search_calls", 0) / row["iterations_done"]
        if row["iterations_done"]
        else None
    )

    artifact_dir = OUTPUT_ROOT / "case_details"
    details_path = artifact_dir / f"{case['case_id']}.json"
    trace = dict(
        case_id=case["case_id"],
        campaign_id=CAMPAIGN_ID,
        instance=path.name,
        equipped_arcs=arcs,
        protocol=settings,
        starts=(
            starts
            if SAVE_SEARCH_TRACES
            else [{k: v for k, v in s.items() if k != "history"} for s in starts]
        ),
    )
    write_json_atomic(details_path, trace)
    row["details_file"] = str(details_path.relative_to(OUTPUT_ROOT))
    return row


def case_row_path(case):
    return OUTPUT_ROOT / "case_rows" / case["experiment"] / f"{case['case_id']}.json"


def error_row(case, error):
    return dict(
        campaign_id=CAMPAIGN_ID,
        case_id=case["case_id"],
        execution_id=case["case_id"],
        experiment=case["experiment"],
        ablation=case["ablation"],
        instance=Path(case["instance_path"]).name,
        n_customers=_extract_n_from_name(Path(case["instance_path"]).name),
        coverage=case["coverage"],
        omega=case["omega"],
        placement_seed=case["placement_seed"],
        solver_seed=case["solver_seed"],
        search_version=SEARCH_VERSION,
        energy_model_version=ENERGY_MODEL_VERSION,
        evaluator_version=EVALUATOR_VERSION,
        error=f"{type(error).__name__}: {error}",
        best_cost=None,
        vehicles=None,
    )


def read_successful_checkpoint(case):
    path = case_row_path(case)
    if not path.exists():
        return None
    row = json.loads(path.read_text(encoding="utf-8"))
    if row.get("campaign_id") != CAMPAIGN_ID or row.get("case_id") != case["case_id"]:
        raise ValueError(f"Checkpoint does not match current campaign: {path}")
    if row.get("error") is None and np.isfinite(row.get("best_cost", float("nan"))):
        if not (OUTPUT_ROOT / row["details_file"]).is_file():
            raise RuntimeError(f"Missing details file for completed case: {path}")
        return row
    return None  # A failed case is retried on resume.


def run_campaign(experiment):
    cases = CAMPAIGN_CASES[experiment]
    raw_file = EXPERIMENT_FILES[experiment]
    rows = []
    print(f"[{experiment}] {len(cases)} cases")
    for index, case in enumerate(cases, 1):
        row = read_successful_checkpoint(case)
        resumed = row is not None
        if row is None:
            try:
                row = solve_overlay_case(case)
            except Exception as exc:
                row = error_row(case, exc)
                write_json_atomic(case_row_path(case), row)
                rows.append(row)
                write_csv_atomic(raw_file, pd.DataFrame(rows))
                if FAIL_FAST:
                    raise RuntimeError(
                        f"Case failed: {case['instance_path']}, {case['ablation']}. "
                        f"Results saved in {OUTPUT_ROOT}. Fix the error and resume. {exc}"
                    ) from exc
            else:
                write_json_atomic(case_row_path(case), row)
                rows.append(row)
        else:
            rows.append(row)
        # Every completed/failed case is checkpointed, not only every placement seed.
        write_csv_atomic(raw_file, pd.DataFrame(rows))
        status = "RESUME" if resumed else ("ERROR" if row.get("error") else "OK")
        print(
            f"  [{index}/{len(cases)}] {Path(case['instance_path']).name} "
            f"gamma={case['coverage']:g}, omega={case['omega']:g}, "
            f"variant={case['ablation']}, seed={case['solver_seed']} | {status} | "
            f"cost={row.get('best_cost')}",
            flush=True,
        )
    if not rows:
        write_csv_atomic(
            raw_file, pd.DataFrame(columns=["case_id", "experiment", "error"])
        )
    return pd.DataFrame(rows)


print("Helpers ready")


In [ ]:
# -------------------------
# Select benchmarks and prepare a new campaign (or resume a matching one)
# -------------------------
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Benchmark directory not found: {DATA_DIR.resolve()}")
if NUM_RUNS < 1 or MAX_ITERATIONS < 1 or MAX_MOVES_PER_ITERATION < 1:
    raise ValueError("Search budgets must be positive.")
if not np.isfinite(INITIAL_SOC_FRACTION) or not 0 <= INITIAL_SOC_FRACTION <= 1:
    raise ValueError("INITIAL_SOC_FRACTION must lie in [0, 1].")
if ABLATION_CASES != [(0.10, 0.9)]:
    raise ValueError("The reported ablation uses gamma=0.1 and omega=0.9 only.")
INSTANCE_PATHS = collect_instances(
    DATA_DIR, INSTANCE_LIMIT, INSTANCE_FILTER, INSTANCE_SIZES
)
INPUT_SHA256 = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in INSTANCE_PATHS
}
ABLATION_CONFIGS = [
    {"ablation": "baseline", "overrides": {}},
    {"ablation": "no_shaking", "overrides": {"shake_tenure": MAX_ITERATIONS + 1}},
    {
        "ablation": "no_extended_operators",
        "overrides": {"use_extended_operators": False},
    },
    {"ablation": "no_candidate_filtering", "overrides": {"use_candidate_lists": False}},
]


def make_case(
    path,
    experiment,
    coverage,
    omega,
    placement_seed,
    solver_seed,
    variant="baseline",
    overrides=None,
):
    payload = dict(
        instance_path=str(path.resolve()),
        experiment=experiment,
        coverage=float(coverage),
        omega=float(omega),
        placement_seed=int(placement_seed),
        solver_seed=int(solver_seed),
        ablation=variant,
        overrides=overrides or {},
    )
    payload["case_id"] = hash_object(
        dict(
            instance=path.name,
            input_sha256=INPUT_SHA256[path.name],
            **{k: v for k, v in payload.items() if k != "instance_path"},
        )
    )
    return payload


CAMPAIGN_CASES = {
    key: []
    for key in [
        "no_wireless_baseline",
        "multiseed_replication",
        "omega_sensitivity",
        "ablation_frvcp",
    ]
}
for path in INSTANCE_PATHS:
    if RUN_NO_WIRELESS_BASELINE:
        CAMPAIGN_CASES["no_wireless_baseline"].append(
            make_case(
                path, "no_wireless_baseline", 0, 0, BASE_SOLVER_SEED, BASE_SOLVER_SEED
            )
        )
    if RUN_MULTI_SEED:
        for placement in MULTISEED_PLACEMENT_SEEDS:
            for gamma in COVERAGE_LEVELS:
                CAMPAIGN_CASES["multiseed_replication"].append(
                    make_case(
                        path,
                        "multiseed_replication",
                        gamma,
                        BASE_OMEGA,
                        placement,
                        BASE_SOLVER_SEED,
                    )
                )
    if RUN_SENSITIVITY:
        for omega in OMEGA_LEVELS:
            for gamma in COVERAGE_LEVELS:
                CAMPAIGN_CASES["omega_sensitivity"].append(
                    make_case(
                        path,
                        "omega_sensitivity",
                        gamma,
                        omega,
                        SENSITIVITY_PLACEMENT_SEED,
                        BASE_SOLVER_SEED,
                    )
                )
    if RUN_ABLATION:
        for gamma, omega in ABLATION_CASES:
            configs = ABLATION_CONFIGS[:]
            if _instance_selection_seed(path.name, BASE_SOLVER_SEED) % 2:
                configs.reverse()
            for config in configs:
                if (
                    config["ablation"] == "no_candidate_filtering"
                    and _extract_n_from_name(path.name) <= 20
                ):
                    continue
                CAMPAIGN_CASES["ablation_frvcp"].append(
                    make_case(
                        path,
                        "ablation_frvcp",
                        gamma,
                        omega,
                        ABLATION_PLACEMENT_SEED,
                        BASE_SOLVER_SEED,
                        config["ablation"],
                        config["overrides"],
                    )
                )

CAMPAIGN_SETTINGS = dict(
    input_sha256=INPUT_SHA256,
    search_sha256=SEARCH_SHA256,
    charging_sha256=CHARGING_SHA256,
    python_version=platform.python_version(),
    frvcpy_version=importlib.metadata.version("frvcpy"),
    numpy_version=np.__version__,
    pandas_version=pd.__version__,
    search_version=SEARCH_VERSION,
    energy_model_version=ENERGY_MODEL_VERSION,
    protocol=protocol_settings(40, BASE_SOLVER_SEED),
    save_search_traces=SAVE_SEARCH_TRACES,
    cases={
        key: [{k: v for k, v in case.items() if k != "instance_path"} for case in cases]
        for key, cases in CAMPAIGN_CASES.items()
    },
)
SETTINGS_FINGERPRINT = hash_object(CAMPAIGN_SETTINGS)
if RESUME_OUTPUT_DIR:
    OUTPUT_ROOT = Path(RESUME_OUTPUT_DIR)
    manifest = json.loads(
        (OUTPUT_ROOT / "campaign_manifest.json").read_text(encoding="utf-8")
    )
    if manifest["settings_fingerprint"] != SETTINGS_FINGERPRINT:
        raise RuntimeError(
            "Resume settings differ. Use RESUME_OUTPUT_DIR=None for a new campaign."
        )
    CAMPAIGN_ID = manifest["campaign_id"]
else:
    CAMPAIGN_ID = uuid.uuid4().hex
    run_label = (
        datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S_UTC_") + CAMPAIGN_ID[:8]
    )
    OUTPUT_ROOT = OUTPUT_PARENT / run_label
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
    manifest = dict(
        campaign_id=CAMPAIGN_ID,
        settings_fingerprint=SETTINGS_FINGERPRINT,
        settings=CAMPAIGN_SETTINGS,
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        platform=platform.platform(),
        executable=sys.executable,
    )
    write_json_atomic(OUTPUT_ROOT / "campaign_manifest.json", manifest)
    provenance = OUTPUT_ROOT / "provenance"
    provenance.mkdir()
    shutil.copy2(search_module.__file__, provenance / "tabu_search_impl.py")
    shutil.copy2(charging_module.__file__, provenance / "frvcp_dwc.py")

EXPERIMENT_FILES = {
    "no_wireless_baseline": OUTPUT_ROOT / "no_wireless_baseline/no_wireless_all.csv",
    "multiseed_replication": OUTPUT_ROOT / "multiseed_replication/multiseed_all.csv",
    "omega_sensitivity": OUTPUT_ROOT / "omega_sensitivity/omega_sensitivity_all.csv",
    "ablation_frvcp": OUTPUT_ROOT / "ablation_frvcp/ablation_frvcp_raw.csv",
}
for experiment, cases in CAMPAIGN_CASES.items():
    if len({c["case_id"] for c in cases}) != len(cases):
        raise ValueError(f"Duplicate requested cases in {experiment}")
print("Campaign output:", OUTPUT_ROOT.resolve())
display(
    pd.DataFrame(
        [dict(experiment=k, planned_solves=len(v)) for k, v in CAMPAIGN_CASES.items()]
    )
)


In [ ]:
# -------------------------
# Experiment 0: No-wireless baseline
# -------------------------
no_wireless_baseline_df = run_campaign("no_wireless_baseline")
display(no_wireless_baseline_df.head())


In [ ]:
# -------------------------
# Experiment 1: Five-seed wireless-placement replication
# -------------------------
multiseed_replication_df = run_campaign("multiseed_replication")
display(multiseed_replication_df.head())


In [ ]:
# -------------------------
# Experiment 2: Wireless-intensity sensitivity
# -------------------------
omega_sensitivity_df = run_campaign("omega_sensitivity")
display(omega_sensitivity_df.head())


In [ ]:
# -------------------------
# Experiment 3: Ablation at gamma = 0.1, omega = 0.9
# -------------------------
ablation_frvcp_df = run_campaign("ablation_frvcp")
display(ablation_frvcp_df.head())


In [ ]:
# -------------------------
# Validate and rebuild tables/figures from the completed campaign
# -------------------------
import subprocess

if all(CAMPAIGN_CASES.values()) and len(INSTANCE_PATHS) == 60:
    subprocess.run(
        [
            sys.executable,
            "scripts/rebuild_results.py",
            "--results",
            str(OUTPUT_ROOT),
            "--output",
            str(OUTPUT_ROOT),
        ],
        check=True,
    )
else:
    print("Partial campaign: full-grid publication checks were not run.")


## Notes

Use `INSTANCE_LIMIT` for a short run. New campaigns are saved under `runs/`.
Set `RESUME_OUTPUT_DIR` to continue an unchanged campaign.
